# 0.8 Adaptation Vulnerability + Frequency Prep

This prep notebook builds a combined adaptation scenario by first applying a
raster-explicit vulnerability reduction and then applying a basin-scale flood
frequency shift to the resulting basin risk table.


In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd

from sovereign.flood import (
    apply_basin_frequency_shift,
    build_masked_vulnerability_scenario_curves,
    build_uniform_frequency_shift_table,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# USER CONFIG
model = "wri"
scenario_name = "pubinf_urban_mask_50pct_nbs_shift_150pct"

# Vulnerability settings
vulnerability_reduction = 0.50
mask_value = 1
mask_path = Path.cwd().parent / "outputs" / "flood" / "adaptation" / "test_mask.tif"
target_sector_label = "Public"
target_exposure_name = "inf_pub_capstock.tif"
paired_public_component_name = "pub_nres"

# Frequency settings
shift_factor = 1.50
degrade_protection = False
n_target_basins = 10


In [4]:
# Paths and baseline inputs
root = Path.cwd().parent
flood_dir = root / "inputs" / "flood" / "maps"
exposure_dir = root / "outputs" / "exposure"
risk_map_dir = root / "outputs" / "flood" / "risk" / "maps"
risk_basin_path = root / "outputs" / "flood" / "risk" / "basins" / f"risk_basins_m-{model}.csv"
vulnerability_path = root / "inputs" / "flood" / "vulnerability" / "jrc_depth_damage.csv"
basin_path = root / "outputs" / "boundaries" / "analysis_basins.gpkg"

scenario_root = root / "outputs" / "flood" / "adaptation" / "combined" / scenario_name
scenario_map_dir = scenario_root / "maps"
scenario_basin_dir = scenario_root / "basins"
scenario_map_dir.mkdir(parents=True, exist_ok=True)
scenario_basin_dir.mkdir(parents=True, exist_ok=True)

flood_dic = {
    5: "UGA_wri-flood_RP5.tif",
    10: "UGA_wri-flood_RP10.tif",
    25: "UGA_wri-flood_RP25.tif",
    50: "UGA_wri-flood_RP50.tif",
    100: "UGA_wri-flood_RP100.tif",
    250: "UGA_wri-flood_RP250.tif",
    500: "UGA_wri-flood_RP500.tif",
    1000: "UGA_wri-flood_RP1000.tif",
}

target_exposure_path = exposure_dir / target_exposure_name
paired_public_baseline_paths = {
    rp: str(risk_map_dir / f"WRI_{rp}_{paired_public_component_name}_cap_damages.tif")
    for rp in flood_dic
}

risk_data = pd.read_csv(risk_basin_path)
risk_data = risk_data.iloc[:, 1:] if str(risk_data.columns[0]).startswith("Unnamed") else risk_data
risk_data["AEP"] = 1 / risk_data["RP"]
risk_data["Pr_L_AEP"] = np.where(risk_data["Pr_L"] == 0, 0, 1 / risk_data["Pr_L"])
risk_data.reset_index(drop=True, inplace=True)

# Example target basins for the frequency shift: first N unique basins
target_basins = sorted(risk_data["HB_L6"].unique())[:n_target_basins]
target_basins[:10]


[1060999120.0,
 1061016240.0,
 1061022540.0,
 1061029030.0,
 1061033480.0,
 1061033490.0,
 1061041420.0,
 1061041490.0,
 1061051360.0,
 1061051510.0]

In [5]:
# Vulnerability curves
vuln_df = pd.read_csv(vulnerability_path)
v_heights = vuln_df["flood_depth"].to_list()
v_inf = vuln_df["africa_infrastructure"].to_list()
v_inf_adapted = (pd.Series(v_inf) * (1 - vulnerability_reduction)).tolist()

baseline_damage_function = [v_heights, v_inf]
adapted_damage_function = [v_heights, v_inf_adapted]


In [6]:
# Step 1: build the vulnerability-adapted basin dataframe
_, vulnerability_risk_df, adapted_raster_paths = build_masked_vulnerability_scenario_curves(
    baseline_risk_df=risk_data,
    basin_path=str(basin_path),
    flood_map_lookup=flood_dic,
    flood_dir=str(flood_dir),
    target_exposure_path=str(target_exposure_path),
    mask_path=str(mask_path),
    baseline_damage_function=baseline_damage_function,
    adapted_damage_function=adapted_damage_function,
    target_sector_label=target_sector_label,
    output_dir=str(scenario_map_dir),
    adapted_component_name_template=f"WRI_{{rp}}_pub_inf_cap_damages.tif",
    combined_sector_name_template=f"WRI_{{rp}}_pub_cap_damages.tif",
    baseline_component_paths_by_rp=paired_public_baseline_paths,
    mask_value=mask_value,
)

vulnerability_risk_df.head()


C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\sovereign-risk\lib\site-packages\rasterstats\io.py:328: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP,component_type,exposure_share
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5,vulnerability_adapted,1.0
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5,vulnerability_adapted,1.0
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5,vulnerability_adapted,1.0
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5,vulnerability_adapted,1.0
4,4,UGA.3_1,Arua,1.061033e+09,2.0,3080.162354,6160.324707,5,Public,0.2,0.5,vulnerability_adapted,1.0


In [7]:
# Step 2: apply the basin-scale frequency shift to the vulnerability scenario
frequency_shift_df = build_uniform_frequency_shift_table(
    return_periods=vulnerability_risk_df["RP"].unique(),
    shift_factor=shift_factor,
)

scenario_risk_df = apply_basin_frequency_shift(
    risk_df=vulnerability_risk_df,
    frequency_shift_df=frequency_shift_df,
    basin_ids=target_basins,
    degrade_protection=degrade_protection,
)

scenario_basin_path = scenario_basin_dir / f"risk_basins_m-{model}.csv"
scenario_risk_df.to_csv(scenario_basin_path, index=False)

print("Scenario basin CSV written to:")
print(scenario_basin_path)
scenario_risk_df.head()


Scenario basin CSV written to:
E:\Projects\sovereign-risk-uga\outputs\flood\adaptation\combined\pubinf_urban_mask_50pct_nbs_shift_150pct\basins\risk_basins_m-wri.csv


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP,component_type,exposure_share
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5.0,Public,0.200000,0.5,frequency_shifted,1.0
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5.0,Public,0.200000,0.5,frequency_shifted,1.0
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,7.5,Public,0.133333,0.5,frequency_shifted,1.0
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,7.5,Public,0.133333,0.5,frequency_shifted,1.0
4,4,UGA.3_1,Arua,1.061033e+09,2.0,3080.162354,6160.324707,7.5,Public,0.133333,0.5,frequency_shifted,1.0


## Notes

This notebook applies the two measures in sequence:

1. vulnerability reduction changes the raster-derived damage magnitudes
2. frequency shifting changes the RP/AEP mapping for selected basins

To customize this scenario, change:

- `scenario_name`
- `vulnerability_reduction`
- `mask_path`
- `target_sector_label`
- `target_exposure_name`
- `shift_factor`
- `target_basins`
- `degrade_protection`
